# Baseline residual-comparison atlases

Generate the finalized five-panel baseline comparison for every reviewed dipper, LTV, and microlensing event in the July 1 Review database. Each baseline is scored independently by the production event detector, and each cohort is written as one multipage PDF.

In [1]:
import json
import re
from pathlib import Path
import sqlite3
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from astropy.coordinates import SkyCoord
from astropy.stats import biweight_location
import astropy.units as u
from IPython.display import display
from matplotlib.backends.backend_pdf import PdfPages

candidate_roots = [Path.cwd().resolve(), *Path.cwd().resolve().parents]
repo_root = next(
    (p for p in candidate_roots if (p / 'pyproject.toml').is_file() and (p / 'malca' / 'core' / 'baseline.py').is_file()),
    None,
)
if repo_root is None:
    raise RuntimeError(f'Could not find the MALCA repository above {Path.cwd()}')
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from malca.core.baseline import (
    global_median_baseline,
    per_camera_gp_baseline,
    per_camera_gp_baseline_masked,
)
from malca.core.utils import clean_lc
from malca.io.lightcurve_io import load_lightcurve_df, to_asassn_algorithm_frame
from malca.plotting.lightcurve_publication import FIG_SINGLE_COL_WIDTH, PUBLICATION_STYLE, finalize_publication_figure
from malca.review.native_lightcurve import resolve_lightcurve_path
from malca.review.store import get_candidate_payload
from malca.stv.events import DEFAULT_BASELINE_KWARGS, score_events_bayesian

JD_OFFSET = 2458000.0
RUN_ROOT = repo_root / 'output' / 'runs' / 'dat3-full-extended_2026-07-01-v4'
RUN_PARAMS = json.loads((RUN_ROOT / 'run_params.json').read_text())
REVIEW_DB = RUN_ROOT / 'review' / 'review.db'
BASE_OUTPUT_DIR = repo_root / 'output' / 'pdf' / 'baseline_methods' / 'main_comparison'
EVENT_CLASSES = {
    'dipper': {'label': 'Dippers', 'folder': 'reviewed_dippers'},
    'ltv': {'label': 'LTVs', 'folder': 'reviewed_ltvs'},
    'microlensing': {'label': 'Microlensing events', 'folder': 'reviewed_microlensing'},
}
CAMERA_PALETTE = (
    '#0072B2', '#009E73', '#56B4E9', '#6A3D9A', '#7F7F7F', '#E69F00',
    '#332288', '#44AA99', '#999933', '#8C6D31', '#333333', '#117733',
)


/opt/homebrew/Caskroom/miniconda/base/envs/malca/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
COHORT_SQL = """
SELECT c.candidate_id, c.asas_sn_id, c.asassn_var_name, c.ra, c.dec
FROM candidates AS c
INNER JOIN reviews AS r ON r.candidate_id = c.candidate_id
WHERE lower(trim(coalesce(r.event_class, ''))) = ?
ORDER BY c.candidate_id
"""

def cohort_output_paths(event_config):
    output_dir = BASE_OUTPUT_DIR / event_config['folder']
    output_dir.mkdir(parents=True, exist_ok=True)
    suffix = event_config['folder']
    return {
        'pdf': output_dir / f'baseline_main_comparison_{suffix}.pdf',
        'manifest': output_dir / 'baseline_main_comparison_manifest.csv',
    }

def load_reviewed_cohort(event_class):
    read_only_uri = f'file:{REVIEW_DB.as_posix()}?mode=ro'
    with sqlite3.connect(read_only_uri, uri=True) as connection:
        cohort = pd.read_sql_query(COHORT_SQL, connection, params=(event_class,))
        resolved_paths = []
        for candidate_id in cohort['candidate_id'].astype(str):
            payload = get_candidate_payload(connection, candidate_id)
            resolved = resolve_lightcurve_path(payload, RUN_ROOT)
            resolved_paths.append(str(resolved.resolve()) if resolved is not None else None)
    cohort['resolved_lightcurve_path'] = resolved_paths
    if cohort['candidate_id'].duplicated().any():
        raise AssertionError(f'Reviewed-{event_class} query returned duplicate candidate IDs.')
    return cohort

cohorts = {event_class: load_reviewed_cohort(event_class) for event_class in EVENT_CLASSES}
for event_class, event_config in EVENT_CLASSES.items():
    print(event_config['label'], len(cohorts[event_class]), cohort_output_paths(event_config)['pdf'])


Dippers 183 /Users/calder/code/malca/output/pdf/baseline_methods/main_comparison/reviewed_dippers/baseline_main_comparison_reviewed_dippers.pdf
LTVs 73 /Users/calder/code/malca/output/pdf/baseline_methods/main_comparison/reviewed_ltvs/baseline_main_comparison_reviewed_ltvs.pdf
Microlensing events 22 /Users/calder/code/malca/output/pdf/baseline_methods/main_comparison/reviewed_microlensing/baseline_main_comparison_reviewed_microlensing.pdf


In [3]:
def global_biweight_baseline(frame, *, c=6.0):
    result = frame.copy().reset_index(drop=True)
    mag = pd.to_numeric(result['mag'], errors='coerce').to_numpy(float)
    error = pd.to_numeric(result['error'], errors='coerce').to_numpy(float)
    finite = np.isfinite(mag)
    if not finite.any():
        raise ValueError('No finite magnitudes for the global biweight baseline.')
    center = float(biweight_location(mag[finite], c=c))
    result['baseline'] = center
    result['resid'] = mag - center
    residual = result['resid'].to_numpy(float)
    residual_finite = residual[np.isfinite(residual)]
    residual_median = float(np.median(residual_finite))
    scatter = float(1.4826 * np.median(np.abs(residual_finite - residual_median)))
    error_median = float(np.nanmedian(error[np.isfinite(error) & (error > 0)]))
    error_safe = np.where(np.isfinite(error) & (error > 0), error, error_median)
    robust_std = max(float(np.sqrt(scatter ** 2 + error_median ** 2)), 1e-6)
    result['sigma_resid'] = residual / robust_std
    result['sigma_eff'] = np.maximum(np.sqrt(error_safe ** 2 + scatter ** 2), 1e-6)
    result['baseline_source'] = 'global_biweight'
    return result

def select_best_band(cleaned):
    bands = pd.to_numeric(cleaned['v_g_band'], errors='coerce')
    counts = bands.value_counts()
    candidates = [band for band in (0.0, 1.0) if counts.get(band, 0) > 0]
    if not candidates:
        raise ValueError('No recognized g or V band observations.')
    band_value = max(candidates, key=lambda band: (int(counts.get(band, 0)), band == 0.0))
    return band_value, {0.0: 'g', 1.0: 'V'}[band_value]

EVENT_SCORING_KWARGS = {
    'p_points': int(RUN_PARAMS['p_points']),
    'mag_points': int(RUN_PARAMS['mag_points']),
    'trigger_mode': str(RUN_PARAMS['trigger_mode']),
    'significance_threshold': float(RUN_PARAMS['significance_threshold']),
    'run_min_points': int(RUN_PARAMS['run_min_points']),
    'max_gap_points': int(RUN_PARAMS['run_max_gap_points']),
    'run_max_gap_days': RUN_PARAMS['run_max_gap_days'],
    'run_min_duration_days': RUN_PARAMS['run_min_duration_days'],
    'compute_event_prob': True,
}

def production_event_mask(lightcurve, baseline_result):
    event_mask = np.zeros(len(lightcurve), dtype=bool)
    counts = {}
    for kind in ('dip', 'jump'):
        score = score_events_bayesian(
            lightcurve,
            kind=kind,
            baseline_func=None,
            df_base=baseline_result,
            logbf_threshold=float(RUN_PARAMS[f'logbf_threshold_{kind}']),
            **EVENT_SCORING_KWARGS,
        )
        indices = np.asarray(score['event_indices'], dtype=int)
        if indices.size:
            event_mask[indices] = True
        counts[kind] = int(indices.size)
    counts['either'] = int(event_mask.sum())
    return event_mask, counts

def prepare_candidate(candidate_row):
    path_value = candidate_row['resolved_lightcurve_path']
    if not path_value or not Path(path_value).is_file():
        raise FileNotFoundError(path_value)
    canonical = load_lightcurve_df(Path(path_value), apply_quality=True)
    cleaned = clean_lc(to_asassn_algorithm_frame(canonical)).reset_index(drop=True)
    band_value, band_label = select_best_band(cleaned)
    bands = pd.to_numeric(cleaned['v_g_band'], errors='coerce')
    lightcurve = cleaned.loc[np.isclose(bands, band_value)].copy().reset_index(drop=True)
    production = per_camera_gp_baseline_masked(lightcurve, **DEFAULT_BASELINE_KWARGS).reset_index(drop=True)
    baseline_results = {
        'Global median': global_median_baseline(lightcurve).reset_index(drop=True),
        'Global biweight': global_biweight_baseline(lightcurve),
        'Per-camera GP': per_camera_gp_baseline(lightcurve, **DEFAULT_BASELINE_KWARGS).reset_index(drop=True),
        'Masked per-camera GP': production,
    }
    event_masks = {}
    event_counts = {}
    for name, result in baseline_results.items():
        event_masks[name], event_counts[name] = production_event_mask(lightcurve, result)
    return lightcurve, band_label, production, baseline_results, event_masks, event_counts


In [4]:
def coordinate_label(candidate_row):
    catalog_name = candidate_row.get('asassn_var_name')
    if pd.notna(catalog_name):
        match = re.search(r'(J\d{6}(?:\.\d+)?[+-]\d{6}(?:\.\d+)?)', str(catalog_name))
        if match:
            return match.group(1)
    ra = pd.to_numeric(candidate_row.get('ra'), errors='coerce')
    dec = pd.to_numeric(candidate_row.get('dec'), errors='coerce')
    if np.isfinite(ra) and np.isfinite(dec):
        coord = SkyCoord(float(ra) * u.deg, float(dec) * u.deg)
        ra_text = coord.ra.to_string(unit=u.hourangle, sep='', precision=2, pad=True)
        dec_text = coord.dec.to_string(unit=u.deg, sep='', precision=1, pad=True, alwayssign=True)
        return f'J{ra_text}{dec_text}'
    return str(candidate_row['candidate_id']).removeprefix('stv_')

def camera_color_map(lightcurve):
    cameras = sorted(lightcurve['camera#'].dropna().unique(), key=str)
    return {camera: CAMERA_PALETTE[i % len(CAMERA_PALETTE)] for i, camera in enumerate(cameras)}

def plot_camera_points(ax, frame, y_col, camera_colors, *, masked_col=None):
    for camera, sub in frame.groupby('camera#', sort=True):
        sub = sub.sort_values('JD')
        x = sub['JD'].to_numpy(float) - JD_OFFSET
        ax.scatter(
            x, sub[y_col], s=5, alpha=1.0, color=camera_colors[camera],
            edgecolors='black', linewidths=0.18,
        )
        if masked_col is not None:
            masked = sub[masked_col].fillna(False).to_numpy(bool)
            if masked.any():
                ax.scatter(
                    x[masked], sub.loc[masked, y_col], s=13, marker='x',
                    color='#b2182b', linewidths=0.7, zorder=5,
                )

def make_comparison_figure(candidate_row, band_label, lightcurve, baseline_results, event_masks, *, figure_height=9.0):
    camera_colors = camera_color_map(lightcurve)
    with plt.rc_context(PUBLICATION_STYLE):
        fig, axes = plt.subplots(
            5, 1, figsize=(FIG_SINGLE_COL_WIDTH, float(figure_height)), sharex=True,
            gridspec_kw={'height_ratios': [1, 1, 1, 1, 1]},
        )
        residual_values = np.concatenate([
            pd.to_numeric(result['resid'], errors='coerce').to_numpy(float)
            for result in baseline_results.values()
        ])
        residual_values = residual_values[np.isfinite(residual_values)]
        raw_values = pd.to_numeric(lightcurve['mag'], errors='coerce').to_numpy(float)
        raw_values = raw_values[np.isfinite(raw_values)]
        raw_center = 0.5 * (float(np.min(raw_values)) + float(np.max(raw_values)))
        raw_half_span = 0.5 * (float(np.max(raw_values)) - float(np.min(raw_values))) * 1.05
        vertical_limit = max(0.15, float(np.max(np.abs(residual_values))) * 1.05, raw_half_span)

        plot_camera_points(axes[0], lightcurve, 'mag', camera_colors)
        axes[0].set_ylim(raw_center + vertical_limit, raw_center - vertical_limit)
        axes[0].set_ylabel(r'$m$ [mag]')
        axes[0].set_title('Raw', fontsize=8, pad=3)

        for ax, (name, result) in zip(axes[1:], baseline_results.items()):
            plot_frame = result.copy()
            plot_frame['event_mask'] = event_masks[name]
            plot_camera_points(ax, plot_frame, 'resid', camera_colors, masked_col='event_mask')
            ax.axhline(0, color='0.25', lw=0.7)
            ax.set_ylim(vertical_limit, -vertical_limit)
            ax.set_ylabel(r'$r$ [mag]')
            ax.set_title(name, fontsize=8, pad=3)

        axes[-1].set_xlabel('JD - 2458000')
        for ax in axes:
            ax.tick_params(direction='in', top=True, right=True)
        fig.suptitle(coordinate_label(candidate_row), fontsize=9)
        finalize_publication_figure(fig, rect=(0, 0, 1, 0.982))
        return fig


In [5]:
all_manifests = {}
failed_cohorts = []

for event_class, event_config in EVENT_CLASSES.items():
    cohort = cohorts[event_class]
    paths = cohort_output_paths(event_config)
    manifest_rows = []
    print(f"\nGenerating {event_config['label']} ({len(cohort)} candidates)")

    with PdfPages(paths['pdf']) as atlas_pdf:
        for position, (_, candidate_row) in enumerate(cohort.iterrows(), start=1):
            candidate_id = str(candidate_row['candidate_id'])
            print(f'[{position:03d}/{len(cohort):03d}] {candidate_id}', end=' ... ')
            try:
                lightcurve, band_label, production, baseline_results, event_masks, event_counts = prepare_candidate(candidate_row)
                fig = make_comparison_figure(candidate_row, band_label, lightcurve, baseline_results, event_masks)
                atlas_pdf.savefig(fig)
                plt.close(fig)
                sources = sorted(production['baseline_source'].dropna().astype(str).unique())
                manifest_rows.append({
                    'event_class': event_class,
                    'candidate_id': candidate_id,
                    'asas_sn_id': candidate_row.get('asas_sn_id'),
                    'lightcurve_path': candidate_row.get('resolved_lightcurve_path'),
                    'band': band_label,
                    'n_points': int(len(lightcurve)),
                    'n_cameras': int(lightcurve['camera#'].nunique()),
                    'n_event_global_median': event_counts['Global median']['either'],
                    'n_event_global_biweight': event_counts['Global biweight']['either'],
                    'n_event_per_camera_gp': event_counts['Per-camera GP']['either'],
                    'n_event_masked_per_camera_gp': event_counts['Masked per-camera GP']['either'],
                    'baseline_sources': ','.join(sources),
                    'status': 'ok',
                    'error': '',
                })
                print(f'{len(lightcurve)} points, {lightcurve["camera#"].nunique()} cameras, {band_label} band')
            except Exception as exc:
                manifest_rows.append({
                    'event_class': event_class,
                    'candidate_id': candidate_id,
                    'asas_sn_id': candidate_row.get('asas_sn_id'),
                    'lightcurve_path': candidate_row.get('resolved_lightcurve_path'),
                    'band': '',
                    'n_points': 0,
                    'n_cameras': 0,
                    'n_event_global_median': 0,
                    'n_event_global_biweight': 0,
                    'n_event_per_camera_gp': 0,
                    'n_event_masked_per_camera_gp': 0,
                    'baseline_sources': '',
                    'status': 'failed',
                    'error': f'{type(exc).__name__}: {exc}',
                })
                print(f'FAILED: {type(exc).__name__}: {exc}')

    manifest = pd.DataFrame(manifest_rows)
    manifest.to_csv(paths['manifest'], index=False)
    all_manifests[event_class] = manifest
    display(manifest['status'].value_counts().rename_axis('status').to_frame('candidates'))
    display(manifest.groupby('band').size().rename('candidates').to_frame())
    print(paths['pdf'])
    print(paths['manifest'])
    failures = manifest.loc[manifest['status'] != 'ok']
    if not failures.empty:
        display(failures)
        failed_cohorts.append((event_class, len(failures)))

if failed_cohorts:
    raise RuntimeError(f'Candidate failures remain: {failed_cohorts}')



Generating Dippers (183 candidates)
[001/183] stv_103079502524 ... 

OMP: Info #276: omp_set_nested routine deprecated, please use omp_set_max_active_levels instead.


740 points, 2 cameras, g band
[002/183] stv_103080542701 ... 

687 points, 2 cameras, g band
[003/183] stv_111669273145 ... 

1514 points, 4 cameras, g band
[004/183] stv_111669291649 ... 

...

620 points, 2 cameras, g band
[022/022] stv_77309955721 ... 

705 points, 2 cameras, g band


,candidates
status,
ok,22


,candidates
band,
g,22


/Users/calder/code/malca/output/pdf/baseline_methods/main_comparison/reviewed_microlensing/baseline_main_comparison_reviewed_microlensing.pdf
/Users/calder/code/malca/output/pdf/baseline_methods/main_comparison/reviewed_microlensing/baseline_main_comparison_manifest.csv
